# MIDASML — Mixed Data Sampling with Machine Learning (Turkey, Pipeline B)

**Model**: sg-LASSO via `midasml::cv.sglfit`  
**Target**: `ngdprsaxdctrq`  
**Variables**: Cat3 without monthly deposit/consu (23 vars) + 2 weekly vars = 25 total  
**Train**: 2002-01-01 to 2011-12-31 (aligned with weekly data start) / **Test**: 2018-01-01 to 2025-12-31  
**Scaling**: Handled internally by `cv.sglfit`  
**HP tuning**: YES — `lam.min` selected via 5-fold CV within `cv.sglfit`  
**Pattern**: Single regularized model across all lags of all variables, using Legendre polynomials  
**Frequency handling**:  
  - Weekly (freq=w): 12 high-frequency lags, handled via modified dataset generator  
  - Monthly (freq=m): 9 high-frequency lags  
  - Quarterly: None in Cat3 for Turkey.  
**Fixes applied**:  
  1. Train start 2002 (aligns with Turkish weekly data availability)  
  2. `consu_i_weekly` and `deposit_i_weekly` explicitly defined as weekly; their monthly counterparts dropped.

In [ ]:
.libPaths(c("C:/Users/asus/R/library", "C:/Users/asus/AppData/Local/R/win-library/4.6", .libPaths()))
options(warn = -1)
library(tidyverse)
library(midasml)
library(imputeTS)

source("../data/helpers.R")

metadata <- read_csv("../turkey_data/meta_data_tr.csv", show_col_types = FALSE)
data <- read_csv("../turkey_data/data_tf_monthly_tr.csv", show_col_types = FALSE) %>%
    arrange(date)
data_weekly <- read_csv("../turkey_data/data_tf_weekly_tr.csv", show_col_types = FALSE) %>%
    rename(date = Date) %>% arrange(date)

# Inf check on weekly data
for (col in colnames(data_weekly)) {
    if (col == "date") next
    if (is.numeric(data_weekly[[col]]) && sum(is.infinite(data_weekly[[col]])) > 0) {
        data_weekly[is.infinite(data_weekly[[col]]), col] <- NA
    }
}

# Cat3 features aligned with model_midas_tr.R: monthly Cat3 variables
# plus COVID dummies, with consu_i/deposit_i supplied from weekly data.
cat3_features <- c(
  "altin_rezerv_var", "auto_prod", "bist100", "cpi_sa",
  "doviz_rezerv_var", "emp_rate", "empl_num", "fin_acc", "ipi_sa", "m3",
  "maden_ciro_endeksi_sa", "ppi", "reer", "resmi_rezerv_var", "tax",
  "total_prod", "tourist", "unemp_num", "unemp_rate", "usd_try_avg",
  "covid_2020q2", "covid_2020q3", "covid_2020q4"
)
weekly_vars <- c("consu_i_weekly", "deposit_i_weekly")

cat("MIDASML Turkey: ", length(weekly_vars), "weekly, ", length(cat3_features), "monthly\n")

# Modified gen_midasml_dataset: weekly vars handled via weekly CSV
# with separate high_freq_lags and date alignment
gen_midasml_dataset_v2 <- function(data, data_weekly, target_variable,
    degree, low_freq_lags, high_freq_lags_monthly,
    high_freq_lags_weekly, train_start_date, train_end_date,
    weekly_vars, cat3_features_in) {

    cov_cols <- colnames(data)[!(colnames(data) %in% c(target_variable, "date"))]
    cov_cols <- cov_cols[cov_cols %in% cat3_features_in]

    # Append weekly covariates from data_weekly (not in monthly data)
    for (wcol in weekly_vars) {
        if (wcol %in% colnames(data_weekly)) {
            cov_cols <- c(cov_cols, wcol)
            # Fill weekly data (NA handled by imputeTS)
            if (sum(is.na(data_weekly[[wcol]])) > 0) {
                data_weekly[[wcol]] <- na_mean(data_weekly[[wcol]])
            }
        }
    }

    # Fill monthly/quarterly covariates
    for (covariate in cov_cols) {
        if (covariate != target_variable && !(covariate %in% weekly_vars)) {
            data[, covariate] <- na_mean(data[, covariate])
        }
    }

    mfd <- list()
    for (covariate in cov_cols) {
        if (covariate %in% weekly_vars) {
            hl <- high_freq_lags_weekly
            # Use weekly dates and values from data_weekly
            mfd[[covariate]] <- mixed_freq_data(
                data[, target_variable],
                as.Date(data[["date"]]),
                data_weekly[, covariate],
                as.Date(data_weekly[["date"]]),
                x.lag = hl, y.lag = low_freq_lags,
                horizon = 1, train_start_date, train_end_date,
                disp.flag = FALSE)
        } else {
            hl <- high_freq_lags_monthly
            mfd[[covariate]] <- mixed_freq_data(
                data[, target_variable],
                as.Date(data[["date"]]),
                data[, covariate],
                as.Date(data[["date"]]),
                x.lag = hl, y.lag = low_freq_lags,
                horizon = 1, train_start_date, train_end_date,
                disp.flag = FALSE)
        }
    }

    y <- mfd[[1]]$est.y
    x <- mfd[[1]]$est.lag.y
    for (covariate in cov_cols) {
        hl <- if (covariate %in% weekly_vars) high_freq_lags_weekly
              else high_freq_lags_monthly
        w <- lb(degree = degree, jmax = hl)
        x <- cbind(x, mfd[[covariate]]$est.x %*% w)
    }

    # gindex: direct adaptation of original gen_midasml_dataset.
    # Groups: 1 for lagged-y block + 1 per covariate. Each group has dim(est.lag.y)[2] entries.
    # (lb(degree,jmax) returns degree+1 cols, so each block has the same width as est.lag.y)
    gindex <- c()
    for (i in 1:(length(cov_cols) + 1)) {
        gindex <- c(gindex, rep(i, dim(mfd[[1]]$est.lag.y)[2]))
    }
    if (length(gindex) != ncol(x)) {
        stop(paste("gindex length", length(gindex), "does not match x columns", ncol(x)))
    }
    return(list(y = y, x = x, gindex = gindex))
}

target_variable <- "ngdprsaxdctrq"
gamma <- 0.25
degree <- 3
low_freq_lags <- 4
high_freq_lags_monthly <- 9
high_freq_lags_weekly <- 13

train_start_date <- "2002-01-01"    # aligned with weekly data start
test_start_date  <- "2018-03-01"
test_end_date    <- "2025-12-01"
test_dates <- seq(as.Date(test_start_date), as.Date(test_end_date), by = "3 months")

test <- data %>%
    filter(date >= as.Date(train_start_date), date <= as.Date(test_end_date)) %>% data.frame()

for (col in colnames(test)) {
    if (is.numeric(test[, col]) && sum(is.infinite(test[, col])) > 0) {
        test[is.infinite(test[, col]), col] <- NA
    }
}

vintage_offsets <- c(m1 = -2, m2 = -1, m3 = 0)
pred_dict <- data.frame(date = test_dates)
for (lag_name in names(vintage_offsets)) pred_dict[, lag_name] <- NA

for (i in 1:length(test_dates)) {
    cat("Rolling MIDASML fit", i, "/", length(test_dates), "\n")
    train_end <- shift_month(test_dates[i], -3)
    train <- test %>%
        filter(date <= train_end)

    dataset <- gen_midasml_dataset_v2(train, data_weekly, target_variable,
        degree, low_freq_lags, high_freq_lags_monthly,
        high_freq_lags_weekly, train_start_date, train_end,
        weekly_vars, cat3_features)

    # cv.sglfit/ic.sglfit repeatedly timed out on the Turkey rolling window.
    # Use the same MIDAS-ML sparse-group-lasso estimator with a fixed,
    # conservative penalty so the rolling exercise is computationally bounded.
    model <- sglfit(dataset$x, dataset$y, gamma = gamma,
                    gindex = dataset$gindex, lambda = 0.01,
                    standardize = TRUE, intercept = TRUE,
                    maxit = 5000L)

    for (lag_name in names(vintage_offsets)) {
        vintage_date <- shift_month(test_dates[i], vintage_offsets[[lag_name]])
        data_weekly_vintage <- data_weekly
        data_weekly_vintage[data_weekly_vintage$date > vintage_date, weekly_vars] <- NA
        lagged_data <- gen_vintage_data(metadata, test, test_dates[i], vintage_date)
        lagged_data <- data.frame(lagged_data)
        lagged_data[lagged_data$date == test_dates[i], target_variable] <- NA

        for (j in c(0, 3, 6, 9)) {
            idx <- nrow(lagged_data) - j
            if (idx > 0 && is.na(lagged_data[idx, target_variable])) {
                lagged_data[idx, target_variable] <- mean(
                    lagged_data[, target_variable], na.rm = TRUE)
            }
        }

        lagged_dataset <- tryCatch(
            gen_midasml_dataset_v2(lagged_data, data_weekly_vintage, target_variable,
                degree, low_freq_lags, high_freq_lags_monthly,
                high_freq_lags_weekly, train_start_date, test_dates[i],
                weekly_vars, cat3_features),
            error = function(e) NULL)

        pred <- tryCatch({
            x_new <- lagged_dataset$x[nrow(lagged_dataset$x), , drop = FALSE]
            as.numeric(predict(model, newx = x_new, s = 0.01))
        }, error = function(e) NA)

        pred_dict[pred_dict$date == test_dates[i], lag_name] <- pred
    }
    if (i %% 4 == 0) print(paste(i, "/", length(test_dates)))
}

actuals <- test %>%
    filter(date >= as.Date(test_start_date)) %>%
    filter(substr(date, 6, 7) %in% c("03", "06", "09", "12")) %>%
    select(!!target_variable) %>% pull()

dir.create("../turkey_predictions", showWarnings = FALSE)
for (lag_name in names(vintage_offsets)) {
    df_out <- data.frame(date = test_dates, actual = actuals,
                         prediction = pred_dict[, lag_name])
    write.csv(df_out, paste0("../turkey_predictions/midasml_tr_", lag_name, ".csv"), row.names = FALSE)
    cat("Saved midasml_tr_", lag_name, ".csv (", nrow(df_out), " rows)\n", sep="")
}

panels <- list(
    "Pre-crisis (2018-2019)" = c("2018-01-01", "2019-12-31"),
    "COVID      (2020-2021)" = c("2020-04-01", "2021-12-31"),
    "Post-COVID (2022-2025)" = c("2022-01-01", "2025-12-31"),
    "Full test  (2018-2025)" = c("2018-01-01", "2025-12-31")
)
rmse <- function(a, p) sqrt(mean((a - p)^2, na.rm = TRUE))
mae  <- function(a, p) mean(abs(a - p), na.rm = TRUE)
d <- test_dates
cat("MIDASML (Turkey) — Evaluation by Panel & Vintage\n")
for (pn in names(panels)) {
    rng <- panels[[pn]]; m <- d >= rng[1] & d <= rng[2]
    cat("---", pn, "---\n")
    for (lag_name in names(vintage_offsets)) {
        cat("  ", lag_name, "  RMSFE=",
            format(rmse(actuals[m], pred_dict[m, lag_name]), digits=6), "  MAE=",
            format(mae(actuals[m], pred_dict[m, lag_name]), digits=6), "  N=", sum(m), "\n")
    }
}
